# VietMedBridge — 03: Kiểm toàn bộ dữ liệu → freeze → chia input embedding

**Runtime CPU, Run all sau notebook 02.** Mặc định lấy build trong
active_data_batch.json. Nếu dùng nhiều acc để tăng tốc, chọn
`BATCH_SOURCE="workers"` và `FREEZE_MODE="worker"`; mỗi worker xử lý
một tập build riêng, checkpoint riêng, không ghi manifest tổng.
Chỉ một acc chạy `FREEZE_MODE="coordinator"` sau khi các worker COMPLETE.
Kiểm mọi official ID/URL, source hash, offsets, parent/child và bảo toàn IDs lỗi.
Global dedup giữ toàn bộ aliases; input model giống hệt chỉ cần encode một lần.
Chuẩn bị input embedding thành các file nhỏ trên disk, chưa nạp model GPU.

Mỗi archive được freeze/checkpoint riêng; batch resume tiếp build chưa xong.
Candidate freeze là mốc integrity, không tự duyệt relevance hoặc human QA.
Source audit/golden thật vẫn cần team review; không tạo nhãn thi.


In [ ]:
import importlib.util
import json
import os
import subprocess
import sys
from pathlib import Path

if not (3, 11) <= sys.version_info[:2] < (3, 14):
    raise RuntimeError(
        "Dùng Python 3.11–3.13. Trong Runtime > Change runtime type, "
        "chọn phiên bản runtime có Python trong khoảng này."
    )
if importlib.util.find_spec("google.colab") is None:
    raise RuntimeError("Notebook này được thiết kế cho Google Colab.")

from google.colab import drive
# Chạy lại cell này sau MỖI runtime mới/restart: package chỉ được cài trong phiên Colab.
drive.mount("/content/drive")

# Giữ cùng DATA_ROOT trong cả bốn notebook.
DATA_ROOT = Path("/content/drive/MyDrive/VietMedBridge/data")
WORK_DIR = Path("/content/vmb_work")  # temp files/spill ở ổ local của Colab
DATA_ROOT.mkdir(parents=True, exist_ok=True)
WORK_DIR.mkdir(parents=True, exist_ok=True)
os.environ["HF_HOME"] = "/content/hf_cache"
os.environ["TOKENIZERS_PARALLELISM"] = "false"

REPO_URL = "https://github.com/Platypus27-coder/VietMedBridge.git"
CODE_REVISION = None  # Có thể đặt full commit SHA; mặc định dùng code_lock đã lưu.
EXPECTED_PIPELINE_API = 2
CHECKOUT = Path("/content/VietMedBridge")
lock_path = DATA_ROOT / "data_processing_code_lock.json"
lock = json.loads(lock_path.read_text()) if lock_path.exists() else {}
if lock and lock.get("repo_url") != REPO_URL:
    raise ValueError("DATA_ROOT đang khóa vào một repo khác.")
if lock and lock.get("pipeline_api") != EXPECTED_PIPELINE_API and not CODE_REVISION:
    raise RuntimeError(
        "DATA_ROOT đang dùng code cũ. Để nâng lên data-v2, đặt CODE_REVISION='main' "
        "ở cell này rồi chạy lại; dùng tên crawl/build run mới. Raw cũ vẫn được giữ."
    )
DATA_WORKFLOW_API = "external-extraction-import-v9-legacy-build-freeze"
upgrade = lock.get("workflow_api") != DATA_WORKFLOW_API
reference = CODE_REVISION or ("main" if upgrade else lock.get("git_commit")) or "main"
if not (CHECKOUT / ".git").exists():
    subprocess.run(["git", "clone", "--no-checkout", REPO_URL, str(CHECKOUT)], check=True)
remote = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "remote", "get-url", "origin"], text=True
).strip()
if remote != REPO_URL:
    raise ValueError("Checkout hiện tại không thuộc repo VietMedBridge.")
subprocess.run(["git", "-C", str(CHECKOUT), "fetch", "--depth", "1", "origin", reference], check=True)
CODE_COMMIT = subprocess.check_output(
    ["git", "-C", str(CHECKOUT), "rev-parse", "FETCH_HEAD"], text=True
).strip()
if "vietmedbridge" in sys.modules and globals().get("_VMB_IMPORTED_COMMIT") != CODE_COMMIT:
    raise RuntimeError("Mã nguồn đổi trong runtime đã import package. Restart session rồi chạy lại notebook.")
subprocess.run(["git", "-C", str(CHECKOUT), "checkout", "--detach", "FETCH_HEAD"], check=True)
subprocess.run(
    [sys.executable, "-m", "pip", "install", "-q", "-e", ".[notebook]"],
    cwd=CHECKOUT,
    check=True,
)
PACKAGE_SOURCE = CHECKOUT / "src"
if not (PACKAGE_SOURCE / "vietmedbridge" / "__init__.py").is_file():
    raise RuntimeError(
        f"Repo checkout thiếu package source: {PACKAGE_SOURCE}; commit={CODE_COMMIT}. "
        "Kiểm tra lại git fetch/checkout ở output của cell Bootstrap."
    )
sys.path.insert(0, str(PACKAGE_SOURCE))
importlib.invalidate_caches()
from vietmedbridge.artifacts import atomic_json, runtime_versions
from vietmedbridge import PIPELINE_API_VERSION
if PIPELINE_API_VERSION != EXPECTED_PIPELINE_API:
    raise RuntimeError("Notebook và package API không khớp; chọn commit data-v2 và restart session.")
_VMB_IMPORTED_COMMIT = CODE_COMMIT
if not lock or CODE_REVISION or upgrade:
    atomic_json(lock_path, {"repo_url": REPO_URL, "git_commit": CODE_COMMIT, "pipeline_api": PIPELINE_API_VERSION, "workflow_api": DATA_WORKFLOW_API})
PIPELINE_CONFIG = json.loads((CHECKOUT / "configs/data_pipeline.json").read_text())
atomic_json(DATA_ROOT / "data_processing_runtime.json", {
    "git_commit": CODE_COMMIT, "python": sys.version, "packages": runtime_versions()
})
print("Code commit:", CODE_COMMIT)
import vietmedbridge
print("Package source:", Path(vietmedbridge.__file__).resolve())
print("Dữ liệu/checkpoint:", DATA_ROOT)


## 1. Đọc build hoàn tất và chạy regression của chunker


In [ ]:
from vietmedbridge.artifacts import read_json, atomic_json, digest_json, code_fingerprint
from vietmedbridge.dataset import load_snapshot, parquet_path
from vietmedbridge.chunks import ChunkConfig, load_bge_tokenizer
from vietmedbridge.golden import run_golden_suite
from vietmedbridge.health import health_report, freeze_candidate
from vietmedbridge.index_inputs import prepare_index_inputs, publish_data_handoff
from vietmedbridge.external_import import list_external_source_batches
sys.path.insert(0, str(CHECKOUT / "scripts"))
from freeze_workers import (
    freeze_batch_signature, freeze_worker_manifest_path,
    partition_freeze_builds, verify_freeze_workers,
    write_freeze_worker_manifest,
)

BUILD_RUN_OVERRIDE = None  # dùng để chủ động đọc build cũ
BATCH_SOURCE = "active_batch" #@param ["active_batch", "workers"]
WORKER_EXPECTED_SOURCE = "" #@param {type:"string"}
FREEZE_MODE = "serial" #@param ["serial", "worker", "coordinator"]
FREEZE_TEAM_SIZE = 3 #@param {type:"integer"}
FREEZE_WORKER_ID = 0 #@param {type:"integer"}
if FREEZE_MODE != "serial" and (BATCH_SOURCE != "workers" or BUILD_RUN_OVERRIDE):
    raise ValueError("Parallel freeze requires BATCH_SOURCE='workers' and no BUILD_RUN_OVERRIDE.")
batch_path = DATA_ROOT / "active_data_batch.json"
if BUILD_RUN_OVERRIDE:
    build_refs = [{"build_run": BUILD_RUN_OVERRIDE}]
    BATCH = {"schema_version": 1, "state": "COMPLETE", "builds": build_refs}
elif BATCH_SOURCE == "workers":
    expected_source = WORKER_EXPECTED_SOURCE.strip() or str(DATA_ROOT / "incoming")
    expected_paths = [str(path.resolve()) for path in list_external_source_batches(
        DATA_ROOT, expected_source)]
    if not expected_paths:
        raise FileNotFoundError(f"Không tìm thấy archive nguồn: {expected_source}")
    expected_set = set(expected_paths)
    gathered = {}
    prior = read_json(batch_path) if batch_path.exists() else {}
    for item in prior.get("builds", []):
        if item.get("state") == "COMPLETE" and item.get("source_path") in expected_set:
            gathered[item["source_path"]] = item
    worker_dir = DATA_ROOT / "worker_batches"
    worker_files = sorted(worker_dir.glob("*.json")) if worker_dir.exists() else []
    for worker_file in worker_files:
        worker = read_json(worker_file)
        worker_items = worker.get("builds", [])
        if not any(item.get("source_path") in expected_set for item in worker_items):
            continue
        if worker.get("state") != "COMPLETE":
            raise RuntimeError(f"Worker chưa hoàn tất: {worker_file.name} ({worker.get('state')})")
        for item in worker_items:
            if item.get("state") != "COMPLETE":
                raise RuntimeError(f"Build worker chưa hoàn tất: {item.get('source_path')}")
            source_path = item.get("source_path")
            if source_path in gathered and (gathered[source_path].get("build_run") != item.get("build_run")
                or gathered[source_path].get("snapshot_sha256") != item.get("snapshot_sha256")):
                raise ValueError(f"Hai worker đã tạo build không khớp cho cùng archive: {source_path}")
            gathered[source_path] = item
    actual_set = set(gathered)
    if actual_set != expected_set:
        missing, extra = sorted(expected_set-actual_set), sorted(actual_set-expected_set)
        raise ValueError(f"Worker coverage chưa khớp batch; missing={missing}, extra={extra}")
    build_refs = [gathered[path] for path in expected_paths]
    BATCH = {"schema_version": 1, "input_kind": "external", "state": "COMPLETE",
        "builds": build_refs, "assembled_from_workers": [path.name for path in worker_files]}
    if FREEZE_MODE == "serial":
        atomic_json(batch_path, BATCH)
    print(f"Đã hợp nhất {len(worker_files)} worker manifest; đủ {len(build_refs)} archive.")
else:
    if batch_path.exists():
        BATCH = read_json(batch_path)
    else:
        active_path = DATA_ROOT / "active_data_build.json"
        active = read_json(active_path) if active_path.exists() else {}
        if active.get("build_run"):
            BATCH = {"schema_version": 1, "state": "COMPLETE", "builds": [{
                "build_run": active["build_run"], "snapshot_sha256": active.get("snapshot_sha256")}]}
        else:
            BATCH = {}
    if BATCH.get("state") != "COMPLETE" or not BATCH.get("builds"):
        raise RuntimeError("Notebook 02 chưa hoàn tất toàn batch. Chạy 02 trước.")
    build_refs = BATCH["builds"]
ACTIVE_BUILDS = []
for item in build_refs:
    run_name = item["build_run"]
    build_dir = DATA_ROOT / "processed" / run_name
    build = read_json(build_dir / "build.json")
    if not build.get("selected_range_complete"):
        raise RuntimeError(f"Build {run_name} còn shard thiếu; resume notebook 02.")
    if item.get("snapshot_sha256") and item["snapshot_sha256"] != build["snapshot_sha256"]:
        raise ValueError(f"Batch pointer và snapshot {run_name} không khớp.")
    config = read_json(build_dir / "config.json")
    ACTIVE_BUILDS.append({"build_run": run_name, "build_dir": build_dir,
                          "build": build, "config": config, "batch_item": item})
BUILD_RUN = ACTIVE_BUILDS[-1]["build_run"]
BUILD_DIR = ACTIVE_BUILDS[-1]["build_dir"]
BUILD = ACTIVE_BUILDS[-1]["build"]
SNAPSHOT = load_snapshot(DATA_ROOT)
OFFICIAL_LINKS = parquet_path(DATA_ROOT, "links_corpus.parquet")
CONFIG = ACTIVE_BUILDS[0]["config"]
for item in ACTIVE_BUILDS[1:]:
    if item["config"]["tokenizer"] != CONFIG["tokenizer"] or item["config"]["chunks"] != CONFIG["chunks"]:
        raise ValueError("Các archive có tokenizer/chunk policy khác nhau; không được gộp âm thầm.")
TOKENIZER, TOKENIZER_SPEC = load_bge_tokenizer(CONFIG["tokenizer"]["revision"])
CHUNK_CONFIG = ChunkConfig(**CONFIG["chunks"])
GOLDEN = run_golden_suite(CHECKOUT / "tests/golden/cases.json", TOKENIZER, TOKENIZER_SPEC, CHUNK_CONFIG)
if not GOLDEN["passed"]:
    raise RuntimeError("Golden regression fail; chưa freeze candidate.")
if FREEZE_MODE != "worker":
    atomic_json(DATA_ROOT / "reports/golden_latest.json", GOLDEN)
print("Batch builds:", len(ACTIVE_BUILDS), "| requested IDs:",
      sum(item["build"]["requested_input_records"] for item in ACTIVE_BUILDS))
print("Freeze mode:", FREEZE_MODE, "| team size:", FREEZE_TEAM_SIZE)


## 2. Kiểm, freeze và chuẩn bị input toàn batch — CPU


In [ ]:
HANDOFFS = []
def freeze_one(item):
    run_name, build_dir, build = item["build_run"], item["build_dir"], item["build"]
    batch_item = item["batch_item"]
    candidate = None
    candidate_name = batch_item.get("candidate_name")
    if candidate_name:
        candidate_path = build_dir / candidate_name
        if candidate_path.is_file():
            saved = read_json(candidate_path)
            saved_sha = digest_json({key:value for key,value in saved.items()
                                     if key != "candidate_manifest_sha256"})
            if (saved.get("state") == "FROZEN_CANDIDATE"
                and saved.get("candidate_manifest_sha256") == saved_sha
                and saved.get("snapshot_sha256") == build["snapshot_sha256"]
                and saved.get("golden", {}).get("code_sha256") == code_fingerprint()):
                candidate = saved
    if candidate is None:
        for path in sorted(build_dir.glob("candidate-*.json")):
            saved = read_json(path)
            saved_sha = digest_json({key:value for key,value in saved.items()
                                     if key != "candidate_manifest_sha256"})
            if (saved.get("state") == "FROZEN_CANDIDATE"
                and saved.get("candidate_manifest_sha256") == saved_sha
                and saved.get("snapshot_sha256") == build["snapshot_sha256"]
                and saved.get("golden", {}).get("code_sha256") == code_fingerprint()):
                candidate, candidate_name = saved, path.name
                break
    if candidate is None:
        crawl_dir = DATA_ROOT / "crawl" / build["crawl_run"] if build.get("crawl_run") else None
        health = health_report(build_dir, official_links=OFFICIAL_LINKS, crawl_dir=crawl_dir,
            work_dir=WORK_DIR, audit_size=PIPELINE_CONFIG["audit_size"])
        candidate = freeze_candidate(build_dir, health, golden_report=GOLDEN)
        candidate_name = f"candidate-{candidate['candidate_manifest_sha256'][:16]}.json"
    inputs = prepare_index_inputs(build_dir, candidate_name, TOKENIZER,
        work_dir=WORK_DIR, part_size=4096)
    record = {"state":"COMPLETE", "snapshot_sha256":build["snapshot_sha256"],
        "candidate_name":candidate_name,
        "candidate_manifest_sha256":candidate["candidate_manifest_sha256"],
        "index_inputs_manifest_sha256":inputs["manifest_sha256"]}
    return candidate, inputs, record

if FREEZE_MODE == "worker":
    if type(FREEZE_TEAM_SIZE) is not int or FREEZE_TEAM_SIZE < 1:
        raise ValueError("FREEZE_TEAM_SIZE must be a positive integer.")
    freeze_refs = [{"build_run":item["build_run"],
        "source_path":item["batch_item"].get("source_path"),
        "snapshot_sha256":item["build"]["snapshot_sha256"]} for item in ACTIVE_BUILDS]
    freeze_code_sha = code_fingerprint()
    freeze_signature = freeze_batch_signature(freeze_refs,
        code_sha256=freeze_code_sha, team_size=FREEZE_TEAM_SIZE)
    assigned = partition_freeze_builds(freeze_refs,
        team_size=FREEZE_TEAM_SIZE, worker_id=FREEZE_WORKER_ID)
    assigned_runs = [item["build_run"] for item in assigned]
    if not assigned_runs:
        raise ValueError("This worker has no assigned builds; reduce FREEZE_TEAM_SIZE.")
    worker_path = freeze_worker_manifest_path(DATA_ROOT,
        batch_signature=freeze_signature, worker_id=FREEZE_WORKER_ID)
    completed = {}
    if worker_path.is_file():
        previous = read_json(worker_path)
        previous_payload = {key:value for key,value in previous.items()
                            if key != "manifest_sha256"}
        if (previous.get("manifest_sha256") != digest_json(previous_payload)
            or previous.get("batch_signature") != freeze_signature
            or previous.get("worker_id") != FREEZE_WORKER_ID
            or previous.get("team_size") != FREEZE_TEAM_SIZE
            or previous.get("assigned_build_runs") != assigned_runs):
            raise ValueError("Freeze worker checkpoint does not match this assignment.")
        completed = previous.get("completed", {})
    write_freeze_worker_manifest(worker_path, batch_signature=freeze_signature,
        code_sha256=freeze_code_sha, team_size=FREEZE_TEAM_SIZE,
        worker_id=FREEZE_WORKER_ID, assigned_build_runs=assigned_runs,
        completed=completed, state="RUNNING")
    by_run = {item["build_run"]:item for item in ACTIVE_BUILDS}
    for index, run_name in enumerate(assigned_runs, 1):
        candidate, inputs, record = freeze_one(by_run[run_name])
        completed[run_name] = record
        write_freeze_worker_manifest(worker_path, batch_signature=freeze_signature,
            code_sha256=freeze_code_sha, team_size=FREEZE_TEAM_SIZE,
            worker_id=FREEZE_WORKER_ID, assigned_build_runs=assigned_runs,
            completed=completed, state="RUNNING")
        print(f"Worker {FREEZE_WORKER_ID}: {index}/{len(assigned_runs)} {run_name} COMPLETE",
              flush=True)
    write_freeze_worker_manifest(worker_path, batch_signature=freeze_signature,
        code_sha256=freeze_code_sha, team_size=FREEZE_TEAM_SIZE,
        worker_id=FREEZE_WORKER_ID, assigned_build_runs=assigned_runs,
        completed=completed, state="COMPLETE")
    print(json.dumps({"state":"FREEZE_WORKER_COMPLETE",
        "worker_id":FREEZE_WORKER_ID, "team_size":FREEZE_TEAM_SIZE,
        "assigned_builds":assigned_runs, "checkpoint":str(worker_path)},
        ensure_ascii=False, indent=2))
elif FREEZE_MODE == "coordinator":
    freeze_refs = [{"build_run":item["build_run"],
        "source_path":item["batch_item"].get("source_path"),
        "snapshot_sha256":item["build"]["snapshot_sha256"]} for item in ACTIVE_BUILDS]
    freeze_code_sha = code_fingerprint()
    freeze_signature = freeze_batch_signature(freeze_refs,
        code_sha256=freeze_code_sha, team_size=FREEZE_TEAM_SIZE)
    records = verify_freeze_workers(DATA_ROOT, freeze_refs,
        batch_signature=freeze_signature, code_sha256=freeze_code_sha,
        team_size=FREEZE_TEAM_SIZE)
    for item, record in zip(ACTIVE_BUILDS, records):
        candidate, inputs = record["candidate"], record["inputs"]
        handoff = publish_data_handoff(DATA_ROOT, item["build_run"], candidate, inputs)
        item["batch_item"].update({"state":"FROZEN",
            "candidate_name":record["candidate_name"],
            "candidate_manifest_sha256":record["candidate_manifest_sha256"],
            "index_inputs_manifest_sha256":record["index_inputs_manifest_sha256"]})
        HANDOFFS.append(handoff)
    BATCH["freeze_state"] = "COMPLETE"
    atomic_json(batch_path, BATCH)
    print(json.dumps({"state":"BATCH_FROZEN", "builds":len(HANDOFFS),
        "documents":sum(item["documents"] for item in HANDOFFS),
        "children":sum(item["children"] for item in HANDOFFS),
        "lineage_state":read_json(DATA_ROOT / "candidate_lineage.json")["state"],
        "lineage_candidates":len(read_json(DATA_ROOT / "candidate_lineage.json")["sources"])},
        ensure_ascii=False, indent=2))
    print("Coordinator đã xác minh checkpoint của mọi worker và công bố batch. Tiếp theo chạy notebook 04.")
else:
    for index, item in enumerate(ACTIVE_BUILDS, 1):
        candidate, inputs, record = freeze_one(item)
        HANDOFF = publish_data_handoff(DATA_ROOT, item["build_run"], candidate, inputs)
        item["batch_item"].update({"state":"FROZEN", "candidate_name":record["candidate_name"],
            "candidate_manifest_sha256":record["candidate_manifest_sha256"],
            "index_inputs_manifest_sha256":record["index_inputs_manifest_sha256"]})
        atomic_json(batch_path, BATCH)
        HANDOFFS.append(HANDOFF)
        print(f"Freeze {index}/{len(ACTIVE_BUILDS)}: {item['build_run']} | documents="
              f"{candidate['counts']['documents']:,} | children={candidate['counts']['children']:,}")
    BATCH["freeze_state"] = "COMPLETE"
    atomic_json(batch_path, BATCH)
    print(json.dumps({"state":"BATCH_FROZEN", "builds":len(HANDOFFS),
        "documents":sum(item["documents"] for item in HANDOFFS),
        "children":sum(item["children"] for item in HANDOFFS),
        "lineage_state":read_json(DATA_ROOT / "candidate_lineage.json")["state"],
        "lineage_candidates":len(read_json(DATA_ROOT / "candidate_lineage.json")["sources"])},
        ensure_ascii=False, indent=2))
    print("Model input parts và checkpoints đã lưu riêng trong từng build. Tiếp theo chạy notebook 04.")


### Chạy song song trên 3 tài khoản

Cả ba tài khoản dùng cùng DATA_ROOT, cùng BATCH_SOURCE/WORKER_EXPECTED_SOURCE,
FREEZE_TEAM_SIZE=3 và FREEZE_MODE="worker"; đặt FREEZE_WORKER_ID lần lượt
0, 1, 2. Các worker nhận 3/2/2 build theo thứ tự ổn định và chỉ ghi vào
thư mục build riêng cùng `data/freeze_workers/<batch-signature>/worker-N.json`.
Mỗi build xong được checkpoint; chạy lại đúng worker ID để resume.
Không chạy hai runtime cùng worker ID.

Khi cả ba manifest báo COMPLETE, một tài khoản đổi sang
FREEZE_MODE="coordinator", giữ FREEZE_TEAM_SIZE=3 và chạy lại notebook.
Coordinator xác minh đủ build/candidate/index-input hashes rồi mới cập nhật
manifest tổng, candidate lineage và active pointer. Không chạy coordinator
khi còn worker đang làm. Notebook này vẫn chạy CPU; embedding dùng GPU ở 04.

Candidate lớn không được nạp vào catalog RAM của pilot. Notebook 04 sẽ
đọc một mẫu từ các input parts để đo BGE/Qwen/reranker và ghi chi phí,
trước khi triển khai inference/index toàn corpus. Nhãn/relevance score không
được suy từ throughput hoặc các kiểm tra integrity này.
